# KV-cache eviction and a planted complaint: Colab T4 runbook

**Runtime → Change runtime type → T4 GPU**, then run top to bottom.

Before you start, upload the repo to `MyDrive/kv-complaints`, **with** `results/manifest.json`
and `data/review_texts.json` from your laptop build and **without** the `yelp/` folder.
Results are written to Drive, so a disconnect loses nothing.

In [14]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/kv-complaints
!ls results/manifest.json data/review_texts.json

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/kv-complaints
data/review_texts.json	results/manifest.json


In [15]:
!pip install -q "kvpress==0.5.5" "transformers==4.57.*" accelerate pytest
!pip freeze > requirements.lock
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "fire>=0.7"

name, memory.total [MiB]
Tesla T4, 15360 MiB
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kvpress 0.5.5 requires fire<0.7,>=0.6.0, but you have fire 0.7.1 which is incompatible.


If an import fails after pip changed numpy or pandas: **Runtime → Restart session**, re-run the first cell, continue here.

## Git
`run.py` records the commit of every run, and Test 4 checks every committed file. Edit your name and email once.

In [5]:
!git config --global --add safe.directory '*'
!git config --global user.name "PranavaMR"
!git config --global user.email "aragornoftirith@gmail.com"
!test -d .git || git init -q
!git add -A && git commit -qm "Freeze code, lexicon and templates (code drafted with Claude)" || true
!git log --oneline | head -3

85948c8 Freeze code, lexicon and templates (code drafted with Claude)


In [11]:
!pip install pipes

ERROR: Could not find a version that satisfies the requirement pipes (from versions: none)
ERROR: No matching distribution found for pipes


## Smoke test: KnormPress at 50% on a ~2k-token dummy context

In [12]:

import sys, shlex
sys.modules['pipes'] = shlex

import torch, kvpress
from kvpress import KnormPress
from run import load_pipe

pipe = load_pipe()
dummy = ' '.join(['The soup was warm and the staff were kind.'] * 200)
print(pipe(dummy, question='\n\nWhat did people say about the soup?',
           press=KnormPress(compression_ratio=0.5), max_new_tokens=40)['answer'])
print('model revision:', pipe.model.config._commit_hash)
del pipe; torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


People said the soup was warm and the staff were kind.
model revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306


## Tests 1 to 4
`-rs` lists skipped tests: all four must say passed, none skipped.

In [13]:
!python -m pytest -q -rs tests


==================================== ERRORS ====================================
____________________ ERROR collecting tests/test_budget.py _____________________
ImportError while importing test module '/content/drive/MyDrive/kv-complaints/tests/test_budget.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_budget.py:3: in <module>
    from audit import RetentionRecorder
audit.py:14: in <module>
    from kvpress import BasePress, ScorerPress
/usr/local/lib/python3.13/dist-packages/kvpress/__init__.py:24: in <module>
    from kvpress.presses.expected_attention_with_stats import ExpectedAttentionStatsPress
/usr/local/lib/python3.13/dist-packages/kvpress/presses/expected_attention_with_stats.py:10: in <module>
    import fire
/usr/local/lib/python3.13/dist-

## Calibration
Full cache, 12 planted contexts, Q1 only. If Full mentions the complaint in under 30% or over 95%,
change `N_RECOMMENDATIONS` in `config.py` **once** (to 8 or 3), log it in `logs/decisions.md`, commit, then launch.

In [ ]:
!python run.py --calibrate

## The 240 runs
Roughly 1 to 1.5 h on a T4. If Colab disconnects, re-run the first cell and this one: it resumes.

In [ ]:
!python run.py

In [ ]:
!wc -l data/runs_raw.jsonl

## Redact, label, analyse
1. `redact.py` writes the committable `results/runs.jsonl`.
2. `labels.py` writes `labels/to_label.csv`. Label every row (1 per aspect the advice addresses, else 0)
   and save it as `labels/labels.csv`. Don't open `labels/key.csv`.
3. Run the analysis, pick the primary detector by your pre-registered rule, set `PRIMARY_DETECTOR`
   in `config.py`, and run the analysis again.

In [ ]:
!python redact.py
!python labels.py

In [ ]:
!python analysis.py

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/fig1_mention_rate.png'))
display(Image('results/figures/fig2_retention.png'))

## Final commit
First clear this notebook's outputs (**Edit → Clear all outputs**, then **File → Save**), so no answer text is saved in it.
Files are staged before the tests, so Test 4 checks everything that is about to be committed.

In [ ]:
!git add -A
!python -m pytest -q -rs tests && git commit -qm "Runs, labels, tables and figures" && git tag v-assessment
!git log --oneline | head -3